# N12 — Overtake Probability Model

**LightGBM binary classifier** — `P(overtake | lap features)`

Trained on the labeled pair dataset produced by N11. Predicts whether car X overtakes car Y **on the current lap**, given the observable state of the pair at lap boundary.

| | |
|---|---|
| Input | `data/processed/overtake_labeled/overtake_pairs_2023_2025.parquet` |
| Train | 2023 + 2024 |
| Test | 2025 |
| Export | `data/models/overtake_probability/` |

### Model choice — LightGBM

This notebook trains a **gradient boosted decision tree** classifier using [LightGBM](https://lightgbm.readthedocs.io/) (Light Gradient Boosting Machine), developed by Microsoft Research ([Ke et al., 2017](https://proceedings.neurips.cc/paper_files/paper/2017/file/6449f44a102fde848669bdd9eb6b76fa-Paper.pdf)).

LightGBM extends classical gradient boosting with two key algorithmic improvements:

- **GOSS (Gradient-based One-Side Sampling):** keeps all samples with large gradients (high error) and randomly drops low-gradient ones, reducing training data without significant information loss.
- **EFB (Exclusive Feature Bundling):** merges mutually exclusive sparse features into single bundles, cutting the effective feature dimensionality.

The result is significantly faster training than XGBoost or standard GBDT with comparable or better accuracy on tabular data.

**Why LightGBM for overtake prediction:**
- Native support for categorical features (`compound_x`, `compound_y`, `circuit_cluster`) without one-hot encoding — it uses optimal histogram splits directly.
- Built-in class imbalance handling via `is_unbalance=True`, avoiding manual oversampling.
- Probabilistic output (`predict_proba`) that the Strategy Agent can consume directly as `P(overtake)`.
- Fast inference — critical when the agent scores dozens of car pairs per simulated lap.


---

## Step 0 — Setup

Standard imports plus LightGBM and SHAP. Paths follow the same convention as N11: `repo_root` resolved via `.git` walker, outputs split between the notebook's `outputs/` folder and `data/models/` for exportable artifacts.


In [22]:
# ── Step 0 · Setup ────────────────────────────────────────────────────────────
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, roc_curve,
    confusion_matrix, classification_report, log_loss
)
from sklearn.calibration import calibration_curve
import shap


ModuleNotFoundError: No module named 'optuna'

In [ ]:
# ── repo root ─────────────────────────────────────────────────────────────────
repo_root = Path().resolve()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent

# ── paths ─────────────────────────────────────────────────────────────────────
OUTPUTS    = repo_root / "notebooks/strategy/overtake_probability/outputs"
PROCESSED  = repo_root / "data/processed/overtake_labeled"
EXPORT_DIR = repo_root / "data/models/overtake_probability"

OUTPUTS.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("repo_root  :", repo_root)
print("OUTPUTS    :", OUTPUTS)
print("PROCESSED  :", PROCESSED)
print("EXPORT_DIR :", EXPORT_DIR)
print("\nlightgbm   :", lgb.__version__)
print("shap       :", shap.__version__)

repo_root  : C:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager
OUTPUTS    : C:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\notebooks\strategy\overtake_probability\outputs
PROCESSED  : C:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\processed\overtake_labeled
EXPORT_DIR : C:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\models\overtake_probability

lightgbm   : 4.6.0
shap       : 0.49.1


---

## Step 1 — Load & Split

The dataset from N11 is split **temporally** — 2023+2024 for training, 2025 for testing. Shuffling is intentionally avoided: the model must generalize to an unseen season, which mirrors the production use case.

Categorical features (`compound_x`, `compound_y`, `circuit_cluster`) are cast to pandas `category` dtype so LightGBM can handle them natively via its histogram-based categorical splits — no one-hot encoding needed.


In [ ]:
# ── Step 1 · Load & split ─────────────────────────────────────────────────────

FEATURES = [
    "gap_ahead_s", "pace_delta_s",
    "tyre_life_x", "tyre_life_y", "tyre_life_diff",
    "speed_trap_delta", "LapNumber", "drs_window",
    "compound_x", "compound_y", "circuit_cluster",
]
CAT_FEATURES = ["compound_x", "compound_y", "circuit_cluster"]
TARGET = "overtake"

In [ ]:
def load_dataset(processed_dir):
    df = pd.read_parquet(processed_dir / "overtake_pairs_2023_2025.parquet")
    for col in CAT_FEATURES:
        df[col] = df[col].astype("category")
    return df


def temporal_split(df):
    train = df[df["Year"].isin([2023, 2024])].copy()
    test  = df[df["Year"] == 2025].copy()
    return train, test


def split_xy(df):
    return df[FEATURES], df[TARGET]


def print_split_stats(train, test):
    for name, subset in [("Train (2023+2024)", train), ("Test  (2025)", test)]:
        n = len(subset)
        ot = subset[TARGET].sum()
        print(f"{name}: {n:,} pairs | {ot:,} overtakes ({ot/n*100:.2f}%)")

In [ ]:
# ── Run ─────────────────────────────────────────────────────────────────────

df = load_dataset(PROCESSED)
train_df, test_df = temporal_split(df)

X_train, y_train = split_xy(train_df)
X_test,  y_test  = split_xy(test_df)

print_split_stats(train_df, test_df)
print(f"\nFeatures : {FEATURES}")
print(f"Cat cols : {CAT_FEATURES}")

Train (2023+2024): 26,063 pairs | 1,733 overtakes (6.65%)
Test  (2025): 14,105 pairs | 830 overtakes (5.88%)

Features : ['gap_ahead_s', 'pace_delta_s', 'tyre_life_x', 'tyre_life_y', 'tyre_life_diff', 'speed_trap_delta', 'LapNumber', 'drs_window', 'compound_x', 'compound_y', 'circuit_cluster']
Cat cols : ['compound_x', 'compound_y', 'circuit_cluster']


---

## Step 2 — Baseline LightGBM

A first model with near-default parameters to establish a performance floor. `is_unbalance=True` lets LightGBM reweight the positive class automatically (equivalent to `scale_pos_weight ≈ 14.7`). We report both AUC-ROC and AUC-PR — the latter is the primary metric given the class imbalance.


In [ ]:
# ── Step 2 · Baseline LightGBM ────────────────────────────────────────────────


PARAMS_BASE = {
    "objective":     "binary",
    "metric":        "binary_logloss",
    "is_unbalance":  True,
    "n_estimators":  500,
    "learning_rate": 0.05,
    "num_leaves":    31,
    "random_state":  42,
    "n_jobs":        -1,
    "verbose":       -1,
}


def train_lgbm(params, X_tr, y_tr, cat_features=CAT_FEATURES):
    model = lgb.LGBMClassifier(**params)
    model.fit(X_tr, y_tr, categorical_feature=cat_features)
    return model


def evaluate(model, X, y, label=""):
    proba = model.predict_proba(X)[:, 1]
    auc_roc = roc_auc_score(y, proba)
    auc_pr  = average_precision_score(y, proba)
    ll      = log_loss(y, proba)
    print(f"{'── ' + label + ' ──':─<40}")
    print(f"  AUC-PR  : {auc_pr:.4f}  (main metric)")
    print(f"  AUC-ROC : {auc_roc:.4f}")
    print(f"  Logloss : {ll:.4f}")
    print(f"\n{classification_report(y, (proba >= 0.5).astype(int), digits=3)}")
    return {"auc_pr": auc_pr, "auc_roc": auc_roc, "logloss": ll}

In [ ]:
# ── main ─────────────────────────────────────────────────────────────────────

model_base = train_lgbm(PARAMS_BASE, X_train, y_train)
metrics_base = evaluate(model_base, X_test, y_test, label="Baseline — Test 2025")


── Baseline — Test 2025 ────────────────
  AUC-PR  : 0.4938  (main metric)
  AUC-ROC : 0.8716
  Logloss : 0.2835

              precision    recall  f1-score   support

           0      0.977     0.895     0.934     13275
           1      0.281     0.659     0.394       830

    accuracy                          0.881     14105
   macro avg      0.629     0.777     0.664     14105
weighted avg      0.936     0.881     0.902     14105



### Step 2 — Baseline: Observations

#### What each metric measures

**AUC-PR (Area Under the Precision-Recall Curve)**
Sweeps every possible decision threshold and plots precision vs recall at each point. The area under that curve summarises performance across all thresholds. For imbalanced datasets this is the metric that matters — it ignores the large pool of true negatives that would inflate AUC-ROC. A random classifier achieves AUC-PR ≈ class prevalence, so here the random baseline is **~0.059** (5.88% positive rate).

**AUC-ROC (Area Under the ROC Curve)**
Probability that the model scores a random positive example higher than a random negative one. Ranges from 0.5 (random) to 1.0 (perfect). Useful as a general ranking metric but optimistic under class imbalance — the large class-0 pool makes it easier to get a high score even with a weak model.

**Log-loss (Binary Cross-Entropy)**
Measures how well the predicted probabilities are calibrated. Penalises confident wrong predictions heavily. Lower is better; a model that always predicts the class prior (~0.06) gives logloss ≈ 0.21, so 0.2835 tells us the model is adding signal but its probabilities may not be perfectly calibrated yet (checked in Step 7).

---

#### Baseline results

| Metric | Value | Reference |
|--------|-------|-----------|
| AUC-PR | **0.4938** | random ≈ 0.059 → **8× above chance** |
| AUC-ROC | **0.8716** | random = 0.500 |
| Log-loss | 0.2835 | prior-only ≈ 0.210 |

A solid floor with zero tuning. The model already ranks overtaking pairs well above non-overtaking ones.

At the default threshold of 0.5, recall on the positive class is **0.659** (captures 2 in 3 actual overtakes) but precision is only **0.281** — many false alarms. This is expected: `is_unbalance=True` makes the model aggressive about flagging positives, and 0.5 is an arbitrary cut-off that doesn't account for the class distribution. The optimal threshold for the Strategy Agent — where missing an overtake is more costly than a false alarm — will likely sit between 0.25 and 0.40. This is analysed in Step 5.


---

## Step 3 — Hyperparameter Search

Optuna study with 30 trials optimising **AUC-PR on a temporal validation split**: 2023 as inner train, 2024 as inner validation. This respects the time ordering — no shuffling, no leakage from future races into the search.

The final model in Step 4 will be retrained on the full 2023+2024 train set using the best parameters found here.


In [ ]:
# ── Step 3 · Hyperparameter search (Optuna) ───────────────────────────────────

# inner split: 2023 train / 2024 val
X_inner_tr = X_train[train_df["Year"] == 2023]
y_inner_tr = y_train[train_df["Year"] == 2023]
X_inner_val = X_train[train_df["Year"] == 2024]
y_inner_val = y_train[train_df["Year"] == 2024]

print(f"Inner train (2023) : {len(X_inner_tr):,} pairs")
print(f"Inner val   (2024) : {len(X_inner_val):,} pairs")


def build_params(trial):
    return {
        "objective":         "binary",
        "metric":            "binary_logloss",
        "is_unbalance":      True,
        "n_estimators":      1000,
        "learning_rate":     trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 20, 120),
        "max_depth":         trial.suggest_int("max_depth", 4, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample":         trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda":        trial.suggest_float("reg_lambda", 0.0, 1.0),
        "random_state":      42,
        "n_jobs":            -1,
        "verbose":           -1,
    }


def objective(trial):
    params = build_params(trial)
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_inner_tr, y_inner_tr,
        categorical_feature=CAT_FEATURES,
        eval_set=[(X_inner_val, y_inner_val)],
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )
    proba = model.predict_proba(X_inner_val)[:, 1]
    return average_precision_score(y_inner_val, proba)


def run_search(n_trials=30):
    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    return study

In [ ]:
# ── main ─────────────────────────────────────────────────────────────────────

study = run_search(n_trials=30)

print(f"\nBest AUC-PR (val 2024) : {study.best_value:.4f}")
print(f"Best params:\n{study.best_params}")
